<a href="https://colab.research.google.com/github/dordor7/SkillReal-Home-Assignment/blob/main/SkillReal_ML_Engineer_Home_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installation of LoHi-Weld
We will clone the repository from GitHub and install the package using pip. We also need to install `rdkit` which is a common dependency for molecular/chemical projects like LoHi.

In [2]:
# Install rdkit (often required for chemical informatics tools like LoHi)
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.5/29.5 MB 37.0 MB/s  0:00:00


In [3]:
import os
import subprocess

# Clone the repository if it doesn't already exist
repo_dir = "/content/LoHi-Weld"
if not os.path.exists(repo_dir):
    print("Cloning LoHi-Weld repository...")
    subprocess.run(["git", "clone", "https://github.com/SylvioBlock/LoHi-Weld.git", repo_dir], check=True)
else:
    print("Repository already cloned.")

# Change working directory to the project folder
os.chdir(repo_dir)
print(f"Current working directory: {os.getcwd()}")

Cloning LoHi-Weld repository...
Current working directory: /content/LoHi-Weld


In [4]:
# Clean requirements.txt of absolute/local conda paths AND remove strict version pinning to prevent compilation issues
import re

with open('requirements.txt', 'r') as f:
    lines = f.readlines()

cleaned_lines = []
for line in lines:
    line_str = line.strip()
    # Skip empty lines or comments
    if not line_str or line_str.startswith('#'):
        continue
    # Skip absolute local paths
    if line_str.startswith('/') or 'file://' in line_str:
        continue

    # Strip out strict '==' version pins so pip fetches modern, pre-compiled wheels compatible with python 3.13
    package_name = re.split(r'==|>=|<=|>|<', line_str)[0].strip()
    if package_name:
        cleaned_lines.append(package_name + '\n')

with open('requirements_cleaned.txt', 'w') as f:
    f.writelines(cleaned_lines)

!pip install -r requirements_cleaned.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.1/221.1 MB 36.4 MB/s  0:00:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 63.7 MB/s  0:00:00
  Attempting uninstall: tbb
    Found existing installation: tbb 2022.3.1
    Uninstalling tbb-2022.3.1:
      Successfully uninstalled tbb-2022.3.1
  Attempting uninstall: mkl
    Found existing installation: mkl 2025.3.1
    Uninstalling mkl-2025.3.1:
      Successfully uninstalled mkl-2025.3.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [mkl-service]


In [5]:
import gdown
import zipfile
import os

# Google Drive file ID from the provided URL
file_id = '1pXeEnREfV_MYcL5MY2vkd9njBm_blPUK'
download_url = f'https://drive.google.com/uc?id={file_id}'
output_zip = 'weld-dataset.zip'

print("Downloading the dataset from Google Drive...")
gdown.download(download_url, output_zip, quiet=False)

print("Extracting the dataset...")
with zipfile.ZipFile(output_zip, 'r') as zip_ref:
    zip_ref.extractall('.')

# Optional: Clean up the downloaded ZIP file
if os.path.exists(output_zip):
    os.remove(output_zip)

print("Dataset download and extraction complete!")

Downloading...
From (original): https://drive.google.com/uc?id=1pXeEnREfV_MYcL5MY2vkd9njBm_blPUK
From (redirected): https://drive.google.com/uc?id=1pXeEnREfV_MYcL5MY2vkd9njBm_blPUK&confirm=t&uuid=1adb66f6-490f-477f-ba54-16bb57d22aa6
To: /content/LoHi-Weld/weld-dataset.zip
100%|██████████| 631M/631M [00:05<00:00, 126MB/s]


Extracting the dataset...
Dataset download and extraction complete!


In [6]:
# Remove annotations for classes other than 'pore' and 'deposit'

import json
from pathlib import Path

root = Path("/content/LoHi-Weld/weld-dataset")

for subset in ("low_resolution_welds", "high_resolution_welds"):
    folder = root / subset

    for path in folder.glob("*.yolo"):
        lines = path.read_text(encoding="utf-8").splitlines()
        kept = [
            line for line in lines
            if line.strip() and line.split()[0] in {"0", "1"}
        ]
        path.write_text("".join(line + "\n" for line in kept), encoding="utf-8")

    for path in folder.glob("*.json"):
        annotation = json.loads(path.read_text(encoding="utf-8"))
        annotation["shapes"] = [
            shape for shape in annotation["shapes"]
            if shape["label"] in {"pore", "deposit"}
        ]
        path.write_text(json.dumps(annotation, indent=2) + "\n", encoding="utf-8")

    print(f"Filtered {subset}")

Filtered low_resolution_welds
Filtered high_resolution_welds


In [7]:
import os
from pathlib import Path

root = Path("/content/LoHi-Weld/weld-dataset")

image_extensions = (".jpg", ".jpeg", ".png", ".bmp")

for subset in ("low_resolution_welds", "high_resolution_welds"):
    folder = root / subset
    if not folder.exists():
        print(f"Folder {folder} does not exist, skipping.")
        continue

    deleted_count = 0
    # Gather all image stems to inspect their annotations
    image_files = [p for p in folder.iterdir() if p.suffix.lower() in image_extensions]

    print(f"--- Cleaning unannotated images in {subset} (starting with {len(image_files)} images) ---")

    for img_path in image_files:
        stem = img_path.stem
        yolo_path = folder / f"{stem}.yolo"
        json_path = folder / f"{stem}.json"

        has_annotations = False

        # Check YOLO file
        if yolo_path.exists():
            lines = yolo_path.read_text(encoding="utf-8").splitlines()
            # If there's at least one non-empty line after filtering, it has annotations
            if any(line.strip() for line in lines):
                has_annotations = True

        # Check JSON file (fallback/double-check)
        if not has_annotations and json_path.exists():
            try:
                import json
                annotation = json.loads(json_path.read_text(encoding="utf-8"))
                if annotation.get("shapes"):
                    has_annotations = True
            except Exception:
                pass

        # If no active annotations, delete the image and its helper files
        if not has_annotations:
            try:
                img_path.unlink()
                if yolo_path.exists():
                    yolo_path.unlink()
                if json_path.exists():
                    json_path.unlink()
                deleted_count += 1
            except Exception as e:
                print(f"Error deleting {stem}: {e}")

    remaining_images = [p for p in folder.iterdir() if p.suffix.lower() in image_extensions]
    print(f"Deleted {deleted_count} unannotated image groups.")
    print(f"Remaining images in {subset}: {len(remaining_images)}\n")

--- Cleaning unannotated images in low_resolution_welds (starting with 2000 images) ---
Deleted 994 unannotated image groups.
Remaining images in low_resolution_welds: 1006

--- Cleaning unannotated images in high_resolution_welds (starting with 1022 images) ---
Deleted 194 unannotated image groups.
Remaining images in high_resolution_welds: 828



In [9]:
import pandas as pd
from pathlib import Path
from collections import Counter

root = Path("/content/LoHi-Weld/weld-dataset")

# Dictionary to hold the final counts
counts = {
    "low": Counter(),
    "high": Counter()
}

# Map internal directory structures to our key names
subsets = {
    "low": root / "low_resolution_welds",
    "high": root / "high_resolution_welds"
}

# Iterate and count annotations from .yolo files
for key, folder in subsets.items():
    if not folder.exists():
        continue
    for yolo_file in folder.glob("*.yolo"):
        try:
            lines = yolo_file.read_text(encoding="utf-8").splitlines()
            for line in lines:
                parts = line.strip().split()
                if parts:
                    cls = parts[0]
                    if cls in {"0", "1"}:
                        counts[key][cls] += 1
        except Exception:
            pass

# Map classes to descriptive names (0 = pore, 1 = deposit)
class_names = {"0": "Pore (Class 0)", "1": "Deposit (Class 1)"}

# Create a structured DataFrame for display
data = {
    "Annotation Class": [class_names["0"], class_names["1"]],
    "Low Resolution Count": [counts["low"]["0"], counts["low"]["1"]],
    "High Resolution Count": [counts["high"]["0"], counts["high"]["1"]],
    "Total": [counts["low"]["0"] + counts["high"]["0"], counts["low"]["1"] + counts["high"]["1"]]
}

summary_annotations_df = pd.DataFrame(data)
display(summary_annotations_df.style.hide(axis='index').set_properties(**{'text-align': 'left'}))

Annotation Class,Low Resolution Count,High Resolution Count,Total
Pore (Class 0),3427,523,3950
Deposit (Class 1),1141,1794,2935


In [11]:
import shutil
from pathlib import Path

root = Path("/content/LoHi-Weld/weld-dataset")
targets = [root / "kfold_high", root / "kfold_low"]

for path in targets:
    if path.exists() and path.is_dir():
        shutil.rmtree(path)
        print(f"Successfully deleted: {path}")
    elif path.exists() and not path.is_dir():
        path.unlink()
        print(f"Successfully deleted file: {path}")
    else:
        print(f"Path does not exist (already deleted): {path}")

Successfully deleted: /content/LoHi-Weld/weld-dataset/kfold_high
Path does not exist (already deleted): /content/LoHi-Weld/weld-dataset/kfold_low


In [15]:
import gdown
import os

# Google Drive IDs for the files
py_file_id = '153boxPkKdP-W39QN9Y_Up7kLD6C2iBsc'
sh_file_id = '1Ylb8qHzA1omFdCgw0sjvtB3hapMGH0xn'

py_download_url = f'https://drive.google.com/uc?id={py_file_id}'
sh_download_url = f'https://drive.google.com/uc?id={sh_file_id}'

target_py = '/content/LoHi-Weld/weld-dataset/create_kfold.py'
target_sh = '/content/LoHi-Weld/weld-dataset/create_kfold.sh'

print("Downloading and replacing create_kfold.py inside weld-dataset...")
gdown.download(py_download_url, target_py, quiet=False)

print("Downloading and replacing create_kfold.sh inside weld-dataset...")
gdown.download(sh_download_url, target_sh, quiet=False)

# Make the shell script executable
if os.path.exists(target_sh):
    os.chmod(target_sh, 0o755)
    print(f"Permissions set: {target_sh} is now executable.")

print("Files successfully placed in weld-dataset!")

Downloading...
From (original): https://drive.google.com/uc?id=153boxPkKdP-W39QN9Y_Up7kLD6C2iBsc
From (redirected): https://drive.google.com/uc?id=153boxPkKdP-W39QN9Y_Up7kLD6C2iBsc&confirm=t&uuid=c475fe98-c822-43e2-bce5-7e4d6938c5d9
To: /content/LoHi-Weld/weld-dataset/create_kfold.py
100%|██████████| 4.35k/4.35k [00:00<00:00, 7.11MB/s]


Downloading...
From (original): https://drive.google.com/uc?id=1Ylb8qHzA1omFdCgw0sjvtB3hapMGH0xn
From (redirected): https://drive.google.com/uc?id=1Ylb8qHzA1omFdCgw0sjvtB3hapMGH0xn&confirm=t&uuid=55bc2f76-a5fe-4247-a5c8-7f4330891e82
To: /content/LoHi-Weld/weld-dataset/create_kfold.sh
100%|██████████| 2.34k/2.34k [00:00<00:00, 4.97MB/s]

Permissions set: /content/LoHi-Weld/weld-dataset/create_kfold.sh is now executable.
Files successfully placed in weld-dataset!


In [16]:
!./weld-dataset/create_kfold.sh

Dataset folder: /content/LoHi-Weld/weld-dataset
Python script: /content/LoHi-Weld/weld-dataset/create_kfold.py

Building kfold_high
kfold_high/test: 166 images (0 repeats), pores: 46 images / 110 boxes, deposits: 157 images / 364 boxes
fold0/train: 882 images (353 repeats), pores: 499 images / 896 boxes, deposits: 499 images / 1149 boxes
fold0/val: 133 images (0 repeats), pores: 37 images / 92 boxes, deposits: 125 images / 281 boxes
fold1/train: 882 images (353 repeats), pores: 499 images / 933 boxes, deposits: 499 images / 1142 boxes
fold1/val: 133 images (0 repeats), pores: 37 images / 94 boxes, deposits: 125 images / 288 boxes
fold2/train: 882 images (352 repeats), pores: 499 images / 939 boxes, deposits: 499 images / 1147 boxes
fold2/val: 132 images (0 repeats), pores: 36 images / 83 boxes, deposits: 125 images / 283 boxes
fold3/train: 882 images (352 repeats), pores: 499 images / 891 boxes, deposits: 499 images / 1144 boxes
fold3/val: 132 images (0 repeats), pores: 36 images / 66 

In [18]:
import yaml
from pathlib import Path

data_dir = Path("/content/LoHi-Weld/data")
yaml_files = list(data_dir.glob("*.yaml")) + list(data_dir.glob("*.yml"))

print(f"Found {len(yaml_files)} YAML config files to update in {data_dir}.")

for yaml_file in yaml_files:
    try:
        # Load the configuration
        with open(yaml_file, 'r', encoding='utf-8') as f:
            config = yaml.safe_load(f) or {}

        # Update class attributes
        config['nc'] = 2
        config['names'] = ['pore', 'deposit']

        # Save the updated configuration back
        with open(yaml_file, 'w', encoding='utf-8') as f:
            yaml.safe_dump(config, f, default_flow_style=False, sort_keys=False)

        print(f"Successfully updated: {yaml_file.name}")
    except Exception as e:
        print(f"Error processing {yaml_file.name}: {e}")

Found 12 YAML config files to update in /content/LoHi-Weld/data.
Successfully updated: high.fold3.yaml
Successfully updated: hyp.data.no.aug.tiny.yaml
Successfully updated: low.fold1.yaml
Successfully updated: hyp.data.aug.tiny.yaml
Successfully updated: low.fold3.yaml
Successfully updated: high.fold1.yaml
Successfully updated: low.fold4.yaml
Successfully updated: high.fold2.yaml
Successfully updated: high.fold0.yaml
Successfully updated: low.fold2.yaml
Successfully updated: low.fold0.yaml
Successfully updated: high.fold4.yaml


In [25]:
import subprocess
from pathlib import Path

google_utils_path = Path("/content/LoHi-Weld/utils/google_utils.py")

# 1. Revert to clean original state
subprocess.run(["git", "checkout", "--", str(google_utils_path)], check=True)
content = google_utils_path.read_text(encoding='utf-8')

# 2. Define a clean, single-line inline safe patch that uses try/except expression style or lambda
old_line = "tag = subprocess.check_output('git tag', shell=True).decode().split()[-1]"
new_line = "tag = (subprocess.check_output('git tag', shell=True).decode().split() or ['v0.1'])[-1]"

if old_line in content:
    content = content.replace(old_line, new_line)
    google_utils_path.write_text(content, encoding='utf-8')
    print("Successfully patched google_utils.py with a robust single-line fallback!")
else:
    print("Target line not found in google_utils.py!")

Successfully patched google_utils.py with a robust single-line fallback!


In [29]:
from pathlib import Path

train_py_path = Path("/content/LoHi-Weld/train.py")
content = train_py_path.read_text(encoding='utf-8')

# We will import torch and allow Model serialization safely at the very top of train.py
safe_imports = """import torch
try:
    from models.yolo import Model
    if hasattr(torch.serialization, 'add_safe_globals'):
        torch.serialization.add_safe_globals([Model])
except Exception:
    pass
"""

# Prepend the safe serialization configurations to train.py
if "add_safe_globals" not in content:
    content = safe_imports + "\n" + content
    train_py_path.write_text(content, encoding='utf-8')
    print("Successfully modified train.py to allow loading YOLOv7 model weights securely!")
else:
    print("train.py already patched.")

Successfully modified train.py to allow loading YOLOv7 model weights securely!


In [32]:
from pathlib import Path

metrics_path = Path("/content/LoHi-Weld/utils/metrics.py")
if metrics_path.exists():
    content = metrics_path.read_text(encoding='utf-8')
    if "np.trapz" in content:
        content = content.replace("np.trapz", "np.trapezoid")
        metrics_path.write_text(content, encoding='utf-8')
        print("Successfully replaced np.trapz with np.trapezoid in metrics.py!")
    else:
        print("np.trapz not found in metrics.py (already replaced?)")
else:
    print(f"Could not locate {metrics_path}")

Successfully replaced np.trapz with np.trapezoid in metrics.py!


In [33]:
# Re-run training smoke test with the patched metrics
!WANDB_MODE=disabled TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python train.py \
  --data data/high.fold0.yaml \
  --cfg cfg/training/yolov7-tiny.yaml \
  --weights yolov7-tiny.pt \
  --hyp data/hyp.data.aug.tiny.yaml \
  --img 640 640 \
  --batch-size 8 --workers 2 --device 0 \
  --epochs 1 --name high.fold0.smoke

I0000 00:00:1794491244.194311   20042 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794491244.269456   20042 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794491246.569084   20042 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794491246.569732   20042 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.